In [ ]:
import sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("student_data.csv")

In [ ]:
df.head()

In [ ]:
df["Passing Status"] = (df["G3"] >= 10).astype(int)

In [ ]:
df.drop(columns=["G3","G2"],axis = 1, inplace = True)

In [ ]:
X = df.drop(columns = ["Passing Status"], axis = 1)
y = df["Passing Status"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42, stratify=y)

In [ ]:
cat_cols = df.select_dtypes("object").columns.to_list()
transformer = ColumnTransformer(
    transformers = [
        ("onehot",OneHotEncoder(drop="first",handle_unknown="ignore"),cat_cols)
    ], remainder= "passthrough" 
)
X_train = transformer.fit_transform(X_train)
X_test = transformer.transform(X_test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
logistic = LogisticRegression(random_state=42,class_weight="balanced")
logistic.fit(X_train_scaled,y_train)
y_pred = logistic.predict(X_test_scaled)
print(accuracy_score(y_test,y_pred))

# Not: Bu yol bana özel. Kullanıcı klasörümdeki Türkçe "İ" harfi SQLite'ta dosya yolu hatası verdiği için
# MLflow veritabanını C:/Users/Public altında tutuyorum. Başka bilgisayarda kendi yolunuzla değiştirin.
mlflow.set_tracking_uri("sqlite:///C:/Users/Public/mlflow.db")

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///C:/Users/Public/mlflow.db")
mlflow.set_experiment("student-predictor")
with mlflow.start_run():
    C = 10
    logistic = LogisticRegression(random_state=42,class_weight="balanced", C=C)
    logistic.fit(X_train_scaled,y_train)
    y_pred = logistic.predict(X_test_scaled)
    accuracy = accuracy_score(y_test,y_pred)
    mlflow.log_param("C", C)
    mlflow.log_metric("accuracy", accuracy)